# Prepare a Microsoft Foundry Model Evaluation

This notebook turns `prepare_foundry_evaluation.py` into a transparent, manually runnable workflow. It validates and explores the 100-task benchmark, creates reusable Foundry evaluation assets, runs comparable model evaluations, and turns the exported outputs into decision-oriented charts.

> **Execution model:** Run cells from top to bottom. Local exploration is safe and free. Cells labeled **Creates Azure assets** or **Billable** only run after you deliberately enable their guard variables.

The evaluation keeps the dataset version, judge deployment, evaluator definitions, prompt template, and generation settings fixed across target models. That control is what makes the comparison meaningful.

## 1. Install and Import Evaluation Dependencies

The kit pins its dependencies in `requirements.txt`. Run the install cell once for the active notebook kernel, then restart the kernel if VS Code asks. The second cell imports the local analysis stack and prints versions so exported results can be reproduced later.

In [ ]:
# Run once in this notebook kernel. This does not contact Azure.
%pip install -r requirements.txt

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

packages = ["azure-ai-projects", "azure-identity", "openai", "pandas", "numpy", "matplotlib", "seaborn"]
versions = {name: importlib.metadata.version(name) for name in packages}
versions["python"] = platform.python_version()
pd.Series(versions, name="version").to_frame()

## 2. Configure Azure AI Foundry Access

Authentication uses `DefaultAzureCredential`, which can reuse an Azure CLI login, managed identity, or another supported credential source. Keep endpoint and deployment values in environment variables; never paste keys or tokens into the notebook.

The project endpoint must include `/api/projects/<project-name>`. Target model values are **deployment names**, not catalog model family names.

In [ ]:
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
JUDGE_DEPLOYMENT = os.getenv("AZURE_AI_JUDGE_DEPLOYMENT")
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID")
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP")
PROJECT_NAME = os.getenv("AZURE_AI_PROJECT_NAME")

TARGET_MODELS = ["DeepSeek-V4-Flash", "gpt-5.4", "gpt-4.1-mini"]
PROFILE = "core"  # Change to "full" to include five safety evaluators.
MAX_COMPLETION_TOKENS = 2048
BEHAVIORAL_THRESHOLD = 4
POLL_SECONDS = 10

# These guards make all remote and billable actions deliberate.
CREATE_FOUNDRY_ASSETS = False
RUN_SMOKE_TEST = False
RUN_FULL_EVALUATION = False

BASE_DIR = Path.cwd()
if not (BASE_DIR / "foundry_eval_dataset.jsonl").exists():
    candidate = BASE_DIR / "kits" / "cost-assessment-kit"
    if (candidate / "foundry_eval_dataset.jsonl").exists():
        BASE_DIR = candidate

DATASET_PATH = BASE_DIR / "foundry_eval_dataset.jsonl"
OUTPUT_DIR = BASE_DIR / "eval_results"

sanitized_config = {
    "project_endpoint_set": bool(PROJECT_ENDPOINT),
    "project_endpoint_host": PROJECT_ENDPOINT.split("/api/projects/")[0] if PROJECT_ENDPOINT else None,
    "judge_deployment": JUDGE_DEPLOYMENT,
    "subscription_id_set": bool(SUBSCRIPTION_ID),
    "resource_group": RESOURCE_GROUP,
    "project_name": PROJECT_NAME,
    "target_models": TARGET_MODELS,
    "profile": PROFILE,
    "dataset_path": str(DATASET_PATH),
    "output_dir": str(OUTPUT_DIR),
}
pd.Series(sanitized_config, name="value").to_frame()

## 3. Load and Validate Evaluation Settings

This preflight mirrors the script's contract before any Azure client is created. It checks local paths, comparison settings, profile values, thresholds, and the presence of remote settings when remote actions are enabled. Failing early prevents incomplete assets and unfair runs.

In [ ]:
assert DATASET_PATH.is_file(), f"Dataset not found: {DATASET_PATH}"
assert PROFILE in {"core", "full"}, "PROFILE must be 'core' or 'full'."
assert TARGET_MODELS and len(TARGET_MODELS) == len(set(TARGET_MODELS)), "Target deployments must be unique."
assert MAX_COMPLETION_TOKENS > 0
assert 1 <= BEHAVIORAL_THRESHOLD <= 5

if CREATE_FOUNDRY_ASSETS or RUN_SMOKE_TEST or RUN_FULL_EVALUATION:
    assert PROJECT_ENDPOINT and "/api/projects/" in PROJECT_ENDPOINT, "Set a valid AZURE_AI_PROJECT_ENDPOINT."
    assert JUDGE_DEPLOYMENT, "Set AZURE_AI_JUDGE_DEPLOYMENT."
if RUN_SMOKE_TEST or RUN_FULL_EVALUATION:
    assert CREATE_FOUNDRY_ASSETS, "Enable CREATE_FOUNDRY_ASSETS before starting runs."

print("Configuration preflight passed.")

## 4. Prepare the Evaluation Dataset

Each JSONL line is already one Foundry item. The validator below parses every line, enforces the required schema, rejects malformed criteria, verifies stable identifiers, and confirms the intentional 10-by-10 category design. A clean copy is written to `eval_results/prepared_dataset.jsonl` for inspection; Azure upload still uses the content-versioned source file.

In [ ]:
REQUIRED_FIELDS = {
    "name", "prompt_id", "category", "difficulty", "query",
    "ground_truth", "expected_behavior", "criteria",
}


def load_and_validate_dataset(path):
    rows = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(f"Invalid JSON on line {line_number}: {error}") from error
        missing = REQUIRED_FIELDS - row.keys()
        if missing:
            raise ValueError(f"Line {line_number} is missing: {sorted(missing)}")
        if not isinstance(row["criteria"], list) or not row["criteria"]:
            raise ValueError(f"Line {line_number} criteria must be a nonempty list")
        valid_criteria = all(
            isinstance(item, dict)
            and all(isinstance(item.get(key), str) and item[key].strip() for key in ("name", "instruction"))
            for item in row["criteria"]
        )
        if not valid_criteria:
            raise ValueError(f"Line {line_number} contains an invalid criterion")
        rows.append(row)

    if len(rows) != 100:
        raise ValueError(f"Expected 100 rows, found {len(rows)}")
    for key in ("name", "prompt_id"):
        if len({row[key] for row in rows}) != len(rows):
            raise ValueError(f"Duplicate {key} values found")
    category_counts = Counter(row["category"] for row in rows)
    if len(category_counts) != 10 or set(category_counts.values()) != {10}:
        raise ValueError(f"Expected ten balanced categories: {dict(category_counts)}")
    return rows


rows = load_and_validate_dataset(DATASET_PATH)
dataset_df = pd.DataFrame(rows)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
prepared_path = OUTPUT_DIR / "prepared_dataset.jsonl"
prepared_path.write_text("\n".join(json.dumps(row, ensure_ascii=True) for row in rows) + "\n", encoding="utf-8")
dataset_version = hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest()[:12]

print(f"Validated {len(dataset_df)} examples. Dataset version: {dataset_version}")
display(dataset_df.head(3))

## 5. Explore Dataset Composition

Balanced counts do not guarantee balanced complexity. These diagnostics reveal missing values, prompt/reference length, criterion density, and category-by-difficulty coverage before model behavior can confound the analysis.

In [ ]:
dataset_df["query_chars"] = dataset_df["query"].str.len()
dataset_df["reference_chars"] = dataset_df["ground_truth"].str.len()
dataset_df["criterion_count"] = dataset_df["criteria"].str.len()

unique_values = dataset_df.apply(
    lambda column: column.map(
        lambda value: json.dumps(value, sort_keys=True) if isinstance(value, (dict, list)) else value
    ).nunique()
)
quality_summary = pd.DataFrame({
    "missing_values": dataset_df.isna().sum(),
    "unique_values": unique_values,
})
display(quality_summary)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.countplot(data=dataset_df, y="category", order=dataset_df["category"].value_counts().index, ax=axes[0, 0], color="#0078D4")
axes[0, 0].set_title("Examples per category")
sns.countplot(data=dataset_df, x="difficulty", order=["easy", "medium", "hard"], ax=axes[0, 1], color="#2D7D46")
axes[0, 1].set_title("Difficulty mix")
sns.histplot(data=dataset_df, x="query_chars", hue="difficulty", bins=20, multiple="stack", ax=axes[1, 0])
axes[1, 0].set_title("Prompt length by difficulty")
coverage = pd.crosstab(dataset_df["category"], dataset_df["difficulty"])
sns.heatmap(coverage, annot=True, fmt="d", cmap="YlGnBu", ax=axes[1, 1])
axes[1, 1].set_title("Category x difficulty coverage")
plt.tight_layout()
plt.show()

display(dataset_df.nlargest(5, "query_chars")[["prompt_id", "category", "difficulty", "query_chars", "query"]])

## 6. Initialize the Model and Evaluators

The core profile combines complementary signals:

- **Coherence** and **relevance** use the fixed judge.
- **Response completeness** compares output with the reference.
- **F1** is deterministic and especially useful for exact-answer tasks.
- **Behavioral adherence** is the primary task-specific evaluator. It checks every row's expected behavior and criteria on a 1-5 scale, passing at 4.
- The optional full profile adds five content-safety evaluators.

Groundedness and fluency are not enabled because this dataset sends only `query` to the model and the repository's comparison contract does not define those metrics. Adding them would change the baseline rather than merely explain it.

In [ ]:
from prepare_foundry_evaluation import (
    build_testing_criteria,
    create_behavioral_evaluator,
    data_source_config,
    model_data_source,
)

assert JUDGE_DEPLOYMENT or not (CREATE_FOUNDRY_ASSETS or RUN_SMOKE_TEST or RUN_FULL_EVALUATION)
if JUDGE_DEPLOYMENT:
    testing_criteria = build_testing_criteria(JUDGE_DEPLOYMENT, "behavioral_adherence", PROFILE)
else:
    # A placeholder lets us inspect evaluator names without remote configuration.
    testing_criteria = build_testing_criteria("<fixed-judge-deployment>", "behavioral_adherence", PROFILE)

metric_catalog = pd.DataFrame([
    {
        "metric": item["name"],
        "evaluator": item["evaluator_name"],
        "judge": item.get("initialization_parameters", {}).get("deployment_name", "deterministic/service"),
        "threshold": item.get("initialization_parameters", {}).get("threshold"),
    }
    for item in testing_criteria
])
display(metric_catalog)

model_plan = pd.DataFrame({
    "deployment": TARGET_MODELS,
    "input_template": "{{item.query}}",
    "max_completion_tokens": MAX_COMPLETION_TOKENS,
    "dataset_version": dataset_version,
})
display(model_plan)

In [ ]:
# CREATES AZURE ASSETS. Set CREATE_FOUNDRY_ASSETS=True in Section 2 first.
if CREATE_FOUNDRY_ASSETS:
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
    openai_client = project_client.get_openai_client()

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    uploaded_dataset = project_client.datasets.upload_file(
        name="foundry-model-benchmark-100",
        version=dataset_version,
        file_path=str(DATASET_PATH),
    )
    custom_evaluator = create_behavioral_evaluator(project_client, "behavioral_adherence")
    evaluation = openai_client.evals.create(
        name=f"foundry-model-comparison-{timestamp}",
        data_source_config=data_source_config(),
        testing_criteria=build_testing_criteria(
            JUDGE_DEPLOYMENT, "behavioral_adherence", PROFILE
        ),
    )
    manifest = {
        "project_endpoint": PROJECT_ENDPOINT,
        "dataset": {"name": uploaded_dataset.name, "version": dataset_version, "id": uploaded_dataset.id},
        "custom_evaluator": {"name": custom_evaluator.name, "version": custom_evaluator.version},
        "evaluation": {"name": evaluation.name, "id": evaluation.id},
        "judge_deployment": JUDGE_DEPLOYMENT,
        "profile": PROFILE,
        "target_models": TARGET_MODELS,
        "runs": [],
    }
    print(f"Created evaluation {evaluation.id} using dataset {uploaded_dataset.id}")
else:
    print("Skipped. Set CREATE_FOUNDRY_ASSETS=True when the configuration is ready.")

## 7. Run a Small Evaluation Smoke Test

A five-row deterministic smoke sample checks authentication, deployment names, model output mapping, evaluator inputs, scores, and error serialization before the 300-response comparison. It creates a separate versioned dataset and runs only the first target model.

> **Billable:** This cell invokes one target model and several judge evaluators for five examples.

In [ ]:
TERMINAL_STATUSES = {"completed", "failed", "canceled", "cancelled"}


def wait_for_run(client, evaluation_id, run, poll_seconds=POLL_SECONDS):
    while run.status not in TERMINAL_STATUSES:
        print(f"{run.name}: {run.status}")
        time.sleep(poll_seconds)
        run = client.evals.runs.retrieve(eval_id=evaluation_id, run_id=run.id)
    return run


if RUN_SMOKE_TEST:
    assert CREATE_FOUNDRY_ASSETS, "Create the reusable assets first."
    smoke_rows = dataset_df.sort_values("prompt_id").groupby("difficulty", group_keys=False).head(2).head(5)
    smoke_path = OUTPUT_DIR / "smoke_dataset.jsonl"
    source_columns = list(rows[0].keys())
    smoke_path.write_text(
        "\n".join(json.dumps(row, ensure_ascii=True) for row in smoke_rows[source_columns].to_dict("records")) + "\n",
        encoding="utf-8",
    )
    smoke_version = hashlib.sha256(smoke_path.read_bytes()).hexdigest()[:12]
    smoke_dataset = project_client.datasets.upload_file(
        name="foundry-model-benchmark-smoke",
        version=smoke_version,
        file_path=str(smoke_path),
    )
    smoke_run = openai_client.evals.runs.create(
        eval_id=evaluation.id,
        name=f"{TARGET_MODELS[0]}-smoke-{timestamp}",
        data_source=model_data_source(smoke_dataset.id, TARGET_MODELS[0], MAX_COMPLETION_TOKENS),
    )
    smoke_run = wait_for_run(openai_client, evaluation.id, smoke_run)
    smoke_items = list(openai_client.evals.runs.output_items.list(eval_id=evaluation.id, run_id=smoke_run.id))
    smoke_output = [item.model_dump(mode="json") for item in smoke_items]
    display(pd.json_normalize(smoke_output).head())
    print(f"Smoke test status: {smoke_run.status}; output items: {len(smoke_output)}")
else:
    print("Skipped. Review Sections 1-6, then set RUN_SMOKE_TEST=True in Section 2.")

## 8. Execute the Full Foundry Evaluation

Each target deployment receives the same 100 queries with the same maximum completion tokens. Runs are intentionally sequential to make throttling and failures visible. After every model, the manifest is checkpointed and completed row-level items are saved, so an interrupted notebook retains useful work.

> **Billable:** Three targets times 100 examples, plus judge calls for each enabled judge-based evaluator. Keep `RUN_FULL_EVALUATION=False` until the smoke output looks correct.

In [ ]:
if RUN_FULL_EVALUATION:
    assert CREATE_FOUNDRY_ASSETS, "Create the reusable assets first."
    manifest_path = OUTPUT_DIR / f"manifest-{timestamp}.json"

    for model in TARGET_MODELS:
        print(f"Starting {model}...")
        started = time.perf_counter()
        try:
            run = openai_client.evals.runs.create(
                eval_id=evaluation.id,
                name=f"{model}-{timestamp}",
                data_source=model_data_source(uploaded_dataset.id, model, MAX_COMPLETION_TOKENS),
            )
            run = wait_for_run(openai_client, evaluation.id, run)
            run_info = {
                "model": model,
                "run_id": run.id,
                "status": run.status,
                "elapsed_seconds": round(time.perf_counter() - started, 1),
                "report_url": getattr(run, "report_url", None),
            }
            if run.status == "completed":
                output_items = list(openai_client.evals.runs.output_items.list(eval_id=evaluation.id, run_id=run.id))
                output_path = OUTPUT_DIR / f"{model}-{timestamp}.json"
                output_path.write_text(
                    json.dumps([item.model_dump(mode="json") for item in output_items], indent=2),
                    encoding="utf-8",
                )
        except Exception as error:
            run_info = {
                "model": model,
                "status": "client_error",
                "elapsed_seconds": round(time.perf_counter() - started, 1),
                "error": f"{type(error).__name__}: {error}",
            }
            print(run_info["error"])

        manifest["runs"].append(run_info)
        manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        print(run_info)

    print(f"Checkpointed manifest: {manifest_path}")
else:
    print("Skipped. Set RUN_FULL_EVALUATION=True only after the smoke test succeeds.")

## 9. Collect and Normalize Evaluation Results

Foundry output items are nested, and field placement can vary slightly across SDK/service versions. This loader recursively finds row metadata, generated text, named metric scores, pass indicators, reasons, and errors. It ignores manifests and prepared datasets. When no cloud outputs exist yet, it returns an empty frame with guidance instead of failing.

In [ ]:
def walk_nodes(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from walk_nodes(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_nodes(child)


def first_value(item, candidate_keys):
    for node in walk_nodes(item):
        for key in candidate_keys:
            if key in node and not isinstance(node[key], (dict, list)):
                return node[key]
    return None


def metric_values(item):
    values = {}
    for node in walk_nodes(item):
        name = node.get("name") or node.get("evaluator_name")
        if not name:
            continue
        score = node.get("score", node.get("result"))
        if isinstance(score, (int, float)) and not isinstance(score, bool):
            values[f"metric.{name}"] = float(score)
        if isinstance(node.get("passed"), bool):
            values[f"passed.{name}"] = node["passed"]
        reason = node.get("reason") or node.get("explanation")
        if isinstance(reason, str):
            values[f"reason.{name}"] = reason
    return values


result_files = [
    path for path in OUTPUT_DIR.glob("*.json")
    if not path.name.startswith("manifest-")
]
normalized_records = []
for path in result_files:
    try:
        items = json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        continue
    if not isinstance(items, list):
        continue
    model = next((name for name in TARGET_MODELS if path.name.startswith(name)), path.stem)
    for item in items:
        record = {
            "model": model,
            "source_file": path.name,
            "prompt_id": first_value(item, ["prompt_id", "name"]),
            "category": first_value(item, ["category"]),
            "difficulty": first_value(item, ["difficulty"]),
            "query": first_value(item, ["query"]),
            "response": first_value(item, ["output_text", "response"]),
            "error": first_value(item, ["error", "error_message"]),
            "latency_ms": first_value(item, ["latency_ms", "duration_ms"]),
        }
        record.update(metric_values(item))
        normalized_records.append(record)

results_df = pd.DataFrame(normalized_records)
if not results_df.empty:
    results_df = results_df.merge(
        dataset_df[["prompt_id", "category", "difficulty", "query"]],
        on="prompt_id", how="left", suffixes=("", "_dataset")
    )
    for column in ("category", "difficulty", "query"):
        results_df[column] = results_df[column].fillna(results_df.pop(f"{column}_dataset"))
    metric_columns = [column for column in results_df if column.startswith("metric.")]
    results_df[metric_columns] = results_df[metric_columns].apply(pd.to_numeric, errors="coerce")
    display(results_df.head())
    print(f"Loaded {len(results_df)} rows from {len(result_files)} result files; metrics: {metric_columns}")
else:
    metric_columns = []
    print("No completed Foundry row-level JSON outputs found yet. Run Sections 6-8, then rerun this cell.")

## 10. Chart Aggregate Evaluation Metrics

Means are useful but can hide instability. The scorecard therefore includes median, standard deviation, evaluated count, a 95% confidence interval for the mean, and pass rate where Foundry supplied a pass flag. The chart ranks model/metric combinations and marks the behavioral-adherence gate at 4/5.

In [ ]:
if metric_columns:
    metric_long = results_df.melt(
        id_vars=["model", "prompt_id", "category", "difficulty"],
        value_vars=metric_columns,
        var_name="metric",
        value_name="score",
    ).dropna(subset=["score"])
    metric_long["metric"] = metric_long["metric"].str.removeprefix("metric.")

    scorecard = metric_long.groupby(["model", "metric"])["score"].agg(
        mean="mean", median="median", std="std", evaluated="count"
    ).reset_index()
    scorecard["ci95"] = 1.96 * scorecard["std"] / np.sqrt(scorecard["evaluated"])

    pass_columns = [column for column in results_df if column.startswith("passed.")]
    pass_rows = []
    for column in pass_columns:
        pass_rows.extend(
            {"model": model, "metric": column.removeprefix("passed."), "pass_rate": values.mean()}
            for model, values in results_df.groupby("model")[column]
            if values.notna().any()
        )
    pass_rates = pd.DataFrame(pass_rows)
    if not pass_rates.empty:
        scorecard = scorecard.merge(pass_rates, on=["model", "metric"], how="left")
    display(scorecard.sort_values(["metric", "mean"], ascending=[True, False]).round(3))

    chart_data = scorecard.copy()
    chart_data["label"] = chart_data["model"] + " | " + chart_data["metric"]
    chart_data = chart_data.sort_values("mean")
    plt.figure(figsize=(11, max(5, 0.35 * len(chart_data))))
    plt.barh(chart_data["label"], chart_data["mean"], xerr=chart_data["ci95"].fillna(0), color="#0078D4", alpha=0.85)
    plt.axvline(BEHAVIORAL_THRESHOLD, color="#D83B01", linestyle="--", label="Behavioral gate (4/5)")
    plt.xlabel("Mean score")
    plt.title("Foundry evaluation scorecard (95% CI)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    scorecard = pd.DataFrame()
    print("Aggregate charts will appear after Foundry result JSON files are available.")

## 11. Compare Performance Across Test Categories

Overall averages can reward a model that excels at easy factual tasks while failing structured output, reasoning, or safety scenarios. The heatmap centers each model/category score against that metric's overall baseline, making regressions and specializations immediately visible.

In [ ]:
if metric_columns:
    primary_metric = "behavioral_adherence" if "metric.behavioral_adherence" in metric_columns else metric_long["metric"].iloc[0]
    primary = metric_long[metric_long["metric"] == primary_metric].copy()
    category_scores = primary.pivot_table(index="category", columns="model", values="score", aggfunc="mean")
    model_baselines = primary.groupby("model")["score"].mean()
    category_delta = category_scores.subtract(model_baselines, axis="columns")

    fig, axes = plt.subplots(1, 2, figsize=(17, 6))
    sns.heatmap(category_scores, annot=True, fmt=".2f", cmap="YlGnBu", vmin=1, vmax=5, ax=axes[0])
    axes[0].set_title(f"Mean {primary_metric} by category")
    sns.heatmap(category_delta, annot=True, fmt="+.2f", cmap="vlag", center=0, ax=axes[1])
    axes[1].set_title("Difference from each model's overall mean")
    plt.tight_layout()
    plt.show()

    difficulty_scores = primary.pivot_table(index="difficulty", columns="model", values="score", aggfunc="mean")
    display(difficulty_scores.round(3))
else:
    category_scores = pd.DataFrame()
    category_delta = pd.DataFrame()
    print("Category comparisons will appear after Foundry results are loaded.")

## 12. Inspect Score Distributions and Metric Relationships

Averages answer "which model scored highest?" Distributions answer "how often did it fail?" Correlations show whether metrics agree or capture different behavior. The second cell adds the kit's existing local benchmark results, making latency, token, and cost charts available even before a Foundry evaluation completes. Cost and latency are supporting operational signals, not substitutes for quality.

In [ ]:
if metric_columns:
    fig, axes = plt.subplots(1, 2, figsize=(17, 6))
    sns.boxplot(data=metric_long, x="score", y="metric", hue="model", ax=axes[0])
    axes[0].set_title("Metric score distributions")

    metric_wide = metric_long.pivot_table(
        index=["model", "prompt_id"], columns="metric", values="score", aggfunc="first"
    )
    sns.heatmap(metric_wide.corr(), annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[1])
    axes[1].set_title("Metric correlation across responses")
    plt.tight_layout()
    plt.show()
else:
    metric_wide = pd.DataFrame()
    print("Distribution and correlation plots will appear after Foundry results are loaded.")

In [ ]:
benchmark_path = BASE_DIR / "results.csv"
if benchmark_path.exists():
    benchmark_df = pd.read_csv(benchmark_path)
    benchmark_summary = benchmark_df.groupby("model").agg(
        prompts=("prompt_id", "count"),
        total_cost_usd=("total_cost_usd", "sum"),
        median_latency_ms=("latency_ms", "median"),
        p95_latency_ms=("latency_ms", lambda values: values.quantile(0.95)),
        mean_tokens=("total_tokens", "mean"),
    ).sort_values("total_cost_usd")
    display(benchmark_summary.round(4))

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    benchmark_summary["total_cost_usd"].plot.bar(ax=axes[0, 0], color="#0078D4", title="Total benchmark cost")
    benchmark_summary[["median_latency_ms", "p95_latency_ms"]].plot.bar(ax=axes[0, 1], color=["#2D7D46", "#D83B01"], title="Median and p95 latency")
    sns.scatterplot(data=benchmark_df, x="latency_ms", y="total_cost_usd", hue="model", size="total_tokens", sizes=(40, 260), ax=axes[1, 0])
    axes[1, 0].set_xscale("log")
    axes[1, 0].set_yscale("log")
    axes[1, 0].set_title("Per-prompt cost vs latency (log scales)")
    benchmark_df.groupby("model")[["input_cost_usd", "output_cost_usd"]].sum().plot.bar(
        stacked=True, ax=axes[1, 1], color=["#50E6FF", "#0078D4"], title="Input/output cost composition"
    )
    for axis in axes.flat:
        axis.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.show()
else:
    benchmark_df = pd.DataFrame()
    benchmark_summary = pd.DataFrame()
    print(f"No local benchmark CSV found at {benchmark_path}")

## 13. Identify Failures and Most Useful Learnings

The most actionable unit is not the overall winner; it is a repeated failure pattern. This section ranks rows below the behavioral gate, charts failures by model and category, and turns the weakest category plus operational extremes into a concise findings table. Review the actual prompts, responses, and judge reasons before changing a prompt or model.

In [ ]:
findings = []
failed_examples = pd.DataFrame()

if metric_columns:
    primary_column = f"metric.{primary_metric}"
    reason_column = f"reason.{primary_metric}"
    results_df["primary_pass"] = results_df[primary_column] >= BEHAVIORAL_THRESHOLD
    failure_columns = [
        column for column in ["model", "prompt_id", "category", "difficulty", "query", "response", primary_column, reason_column, "error"]
        if column in results_df.columns
    ]
    failed_examples = results_df.loc[~results_df["primary_pass"], failure_columns].sort_values(primary_column)
    display(failed_examples.head(20))

    failure_groups = failed_examples.groupby(["model", "category"]).size().rename("failures").reset_index()
    if not failure_groups.empty:
        plt.figure(figsize=(12, 6))
        sns.barplot(data=failure_groups, x="failures", y="category", hue="model")
        plt.title(f"Failures below {BEHAVIORAL_THRESHOLD}/5 by category")
        plt.tight_layout()
        plt.show()

    for model, model_rows in primary.groupby("model"):
        category_means = model_rows.groupby("category")["score"].mean().sort_values()
        failure_rate = (model_rows["score"] < BEHAVIORAL_THRESHOLD).mean()
        findings.append({
            "type": "quality",
            "model": model,
            "observation": f"Weakest category: {category_means.index[0]}",
            "evidence": f"Mean {primary_metric} {category_means.iloc[0]:.2f}; failure rate {failure_rate:.1%}",
            "action": "Review failed rows and judge reasons; test a targeted prompt or model change on this category.",
        })

if not benchmark_summary.empty:
    cheapest = benchmark_summary["total_cost_usd"].idxmin()
    fastest = benchmark_summary["median_latency_ms"].idxmin()
    findings.extend([
        {
            "type": "cost", "model": cheapest, "observation": "Lowest total benchmark cost",
            "evidence": f"${benchmark_summary.loc[cheapest, 'total_cost_usd']:.6f} for {int(benchmark_summary.loc[cheapest, 'prompts'])} prompts",
            "action": "Use as the cost baseline, then confirm it clears category-level quality gates.",
        },
        {
            "type": "latency", "model": fastest, "observation": "Lowest median benchmark latency",
            "evidence": f"{benchmark_summary.loc[fastest, 'median_latency_ms']:.0f} ms median; {benchmark_summary.loc[fastest, 'p95_latency_ms']:.0f} ms p95",
            "action": "Check tail latency and quality failures before selecting it for interactive workloads.",
        },
    ])

findings_df = pd.DataFrame(findings)
display(findings_df)

## 14. Export Results, Charts, and Findings

The export creates a timestamped analysis bundle without overwriting prior runs. It includes normalized row-level data, aggregates, failed examples, machine-readable findings, the benchmark summary, and regenerated publication-ready chart images when their source data exists.

**Interpretation and reproducibility checklist**

- Confirm every model used the same dataset hash, evaluator versions, judge, prompt template, and generation settings.
- Prefer behavioral adherence for task success; treat F1 as narrow evidence for exact-answer tasks.
- Inspect all failures and reasons. A judge disagreement is not automatically a target-model failure.
- Compare categories and difficulty before choosing a model from an overall mean.
- Treat latency and cost as constraints after establishing the required quality floor.
- Record deployment versions and pricing date outside the response data when making a production decision.
- Re-run a held-out set after prompt optimization; do not promote changes using only their training examples.

In [ ]:
export_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
export_dir = OUTPUT_DIR / f"analysis-{export_timestamp}"
export_dir.mkdir(parents=True, exist_ok=False)

(dataset_df.drop(columns=["query_chars", "reference_chars", "criterion_count"], errors="ignore")) \
    .to_json(export_dir / "prepared_dataset.jsonl", orient="records", lines=True)
if not results_df.empty:
    results_df.to_csv(export_dir / "normalized_results.csv", index=False)
if not scorecard.empty:
    scorecard.to_csv(export_dir / "metric_scorecard.csv", index=False)
if not failed_examples.empty:
    failed_examples.to_csv(export_dir / "failed_examples.csv", index=False)
if not benchmark_summary.empty:
    benchmark_summary.to_csv(export_dir / "benchmark_summary.csv")
findings_df.to_json(export_dir / "findings.json", orient="records", indent=2)

if not scorecard.empty:
    chart_data = scorecard.assign(label=lambda frame: frame["model"] + " | " + frame["metric"]).sort_values("mean")
    fig, ax = plt.subplots(figsize=(11, max(5, 0.35 * len(chart_data))))
    ax.barh(chart_data["label"], chart_data["mean"], xerr=chart_data["ci95"].fillna(0), color="#0078D4")
    ax.axvline(BEHAVIORAL_THRESHOLD, color="#D83B01", linestyle="--")
    ax.set(title="Foundry evaluation scorecard", xlabel="Mean score")
    fig.tight_layout()
    fig.savefig(export_dir / "aggregate_metrics.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

if not category_scores.empty:
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(category_scores, annot=True, fmt=".2f", cmap="YlGnBu", vmin=1, vmax=5, ax=ax)
    ax.set_title(f"Mean {primary_metric} by category")
    fig.tight_layout()
    fig.savefig(export_dir / "category_heatmap.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

if not benchmark_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.scatterplot(data=benchmark_df, x="latency_ms", y="total_cost_usd", hue="model", size="total_tokens", sizes=(40, 260), ax=ax)
    ax.set(xscale="log", yscale="log", title="Per-prompt cost vs latency")
    fig.tight_layout()
    fig.savefig(export_dir / "cost_vs_latency.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

reproducibility = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_sha256": hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest(),
    "dataset_version": dataset_version,
    "profile": PROFILE,
    "judge_deployment": JUDGE_DEPLOYMENT,
    "target_models": TARGET_MODELS,
    "max_completion_tokens": MAX_COMPLETION_TOKENS,
    "package_versions": versions,
}
(export_dir / "reproducibility.json").write_text(json.dumps(reproducibility, indent=2), encoding="utf-8")

if CREATE_FOUNDRY_ASSETS:
    openai_client.close()
    project_client.close()
    credential.close()

print(f"Exported analysis bundle to: {export_dir}")
print("Files:", [path.name for path in sorted(export_dir.iterdir())])